# 03 — Outer Vector Add

## Background

Many deep learning operations have a natural **2D structure**: batch × sequence length, rows × columns, etc. Fully exploiting parallelism requires launching work across two dimensions simultaneously — a 2D kernel grid.

Outer vector addition is the most fundamental form of 2D broadcasting, and the building block for understanding operations like positional encoding addition and bias broadcasting.

**Memory layout**: 2D tensors are stored row-major. `C[i, j]` lives at byte offset `(i * M + j) * sizeof(dtype)`.

## Problem Definition

Given 1D tensors `A [N]` and `B [M]`, compute 2D tensor `C [N, M]`:

$$C[i, j] = A[i] + B[j], \quad i \in [0, N),\ j \in [0, M)$$

Equivalent NumPy expression: `C = A[:, None] + B[None, :]`

**Parallel strategy**: launch a 2D grid `(N//BN, M//BM)`; each block owns a `BN × BM` output tile.

In [ ]:
import tilelang
import tilelang.language as T
import triton
import triton.language as tl
import torch

tilelang.disable_cache()
print("TileLang:", tilelang.__version__)
print("Triton:  ", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

## PyTorch Reference

In [ ]:
N, M = 8192, 4096
A = torch.randn(N, dtype=torch.float16, device="cuda")
B = torch.randn(M, dtype=torch.float16, device="cuda")

def ref_outer_add(A, B):
    return A[:, None] + B[None, :]   # [N, M]

C_ref = ref_outer_add(A, B)
print(f"A: {A.shape}, B: {B.shape} → C: {C_ref.shape}")

## Triton Implementation

Triton uses `tl.program_id(0/1)` for the 2D block coordinates. Broadcasting `a[:, None]` and `b[None, :]` constructs the 2D output tile in registers, then a single `tl.store` writes the whole `BN × BM` block.

In [ ]:
@triton.jit
def _outer_add_kernel(A_ptr, B_ptr, C_ptr, N, M,
                      BN: tl.constexpr, BM: tl.constexpr):
    pid_n = tl.program_id(0)
    pid_m = tl.program_id(1)
    rows  = pid_n * BN + tl.arange(0, BN)   # [BN]
    cols  = pid_m * BM + tl.arange(0, BM)   # [BM]
    mask  = (rows[:, None] < N) & (cols[None, :] < M)
    a = tl.load(A_ptr + rows, mask=rows < N, other=0.0)  # [BN]
    b = tl.load(B_ptr + cols, mask=cols < M, other=0.0)  # [BM]
    c = a[:, None] + b[None, :]              # [BN, BM]
    tl.store(C_ptr + rows[:, None] * M + cols[None, :], c, mask=mask)

# ── GPU-adaptive Triton block shape ──────────────────────────────────────────
# gfx1100 / gfx1201: BN=64, BM=64 — balanced 2D tile, fast discrete memory
# gfx1151 (iGPU, unified memory): BN=1, BM=2048 — one row per program.
#   BN=64 causes strided writes (stride=M=8KB) → cache miss on slow unified memory.
#   BN=1: all 1024 Triton threads write to the same row consecutively → coalesced.
arch_tri = torch.cuda.get_device_properties(0).gcnArchName
if arch_tri.startswith("gfx1151"):
    BN_t, BM_t = 1, 2048
else:
    BN_t, BM_t = 64, 64

def triton_outer_add(A, B):
    C = torch.empty(N, M, dtype=A.dtype, device=A.device)
    grid = (triton.cdiv(N, BN_t), triton.cdiv(M, BM_t))
    _outer_add_kernel[grid](A, B, C, N, M, BN=BN_t, BM=BM_t)
    return C

C_tri = triton_outer_add(A, B)
print("Triton correctness:", "✅ PASSED" if torch.allclose(C_ref, C_tri, atol=1e-2) else "❌ FAILED")

## TileLang Implementation

`T.Kernel(D0, D1)` declares a 2D grid; `T.Parallel(BN, BM)` unfolds the 2D parallel loop — closer to Python syntax than Triton.

In [ ]:
# ── GPU-adaptive config ──────────────────────────────────────────────────────
# gfx1100 (RX 7900 XTX): BN=512, BM=64, TH=128 → +131.3% vs PT (sweep winner)
#   Asymmetric tile: tall-in-N (512) × narrow-in-M (64), row-major writes coalesced.
# gfx1201 (R9700): BN=512, BM=256, TH=128 → +259.0% vs PT (sweep winner)
# gfx1151 (Radeon 8060S): RDNA3.5 iGPU, 40 CU, unified memory.
#   Key insight: T.Parallel(BN, BM) with large BN causes strided writes across rows
#   (stride = M = 8KB) → cache miss per thread. Fix: BN=1, BM=M (process 1 row per block,
#   all threads write consecutive columns → fully coalesced). Grid: (N, M//BM).
arch = torch.cuda.get_device_properties(0).gcnArchName

if arch.startswith("gfx1151"):
    # gfx1151: BN=1, BM=4096 (full row per block, coalesced writes)
    @tilelang.jit(pass_configs={
        tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
        tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
    })
    def tl_outer_add(A, B, BLOCK_M: int):
        N, M = T.const("N, M")
        dtype = T.float16
        A: T.Tensor((N,), dtype);  B: T.Tensor((M,), dtype)
        C = T.empty((N, M), dtype)
        with T.Kernel(N, M // BLOCK_M, threads=256) as (row, pid_m):
            for j in T.Parallel(BLOCK_M):
                C[row, pid_m * BLOCK_M + j] = A[row] + B[pid_m * BLOCK_M + j]
        return C

    k = tl_outer_add.compile(N=N, M=M, BLOCK_M=4096)

else:
    BN, BM = (512, 256) if arch.startswith("gfx1201") else (512, 64)

    @tilelang.jit(pass_configs={
        tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
        tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
    })
    def tl_outer_add(A, B, BLOCK_N: int, BLOCK_M: int):
        N, M = T.const("N, M")
        dtype = T.float16
        A: T.Tensor((N,), dtype);  B: T.Tensor((M,), dtype)
        C = T.empty((N, M), dtype)
        with T.Kernel(N // BLOCK_N, M // BLOCK_M, threads=256) as (pid_n, pid_m):
            for i, j in T.Parallel(BLOCK_N, BLOCK_M):
                row = pid_n * BLOCK_N + i
                col = pid_m * BLOCK_M + j
                C[row, col] = A[row] + B[col]
        return C

    k = tl_outer_add.compile(N=N, M=M, BLOCK_N=BN, BLOCK_M=BM)

C_tl = k(A.clone(), B.clone())
print("TileLang correctness:", "✅ PASSED" if torch.allclose(C_ref, C_tl, atol=1e-2) else "❌ FAILED")

## Performance Comparison

In [ ]:
import matplotlib.pyplot as plt

WARMUP, REPEAT = 20, 200

def bench(fn, args, warmup=WARMUP, repeat=REPEAT):
    for _ in range(warmup): fn(*args)
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()
    for _ in range(repeat): fn(*args)
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / repeat

results = {
    "PyTorch\n(broadcast)": bench(ref_outer_add,    [A, B]),
    "Triton\n(2D grid)":    bench(triton_outer_add, [A, B]),
    "TileLang\n(2D grid)":  bench(k,               [A, B]),
}

for name, ms in results.items():
    bw = (N*2 + M*2 + N*M*2) / ms / 1e9
    print(f"  {name.replace(chr(10),' '):22s}: {ms:.4f} ms  ({bw:.2f} TB/s)")

colors = ["#4C72B0", "#55A868", "#C44E52"]
labels = list(results.keys())
values = list(results.values())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax = axes[0]
bars = ax.bar(labels, values, color=colors, width=0.45, edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.01,
            f"{val:.4f} ms", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Latency (ms)"); ax.set_title(f"Outer Add Latency  (N={N}, M={M}, {arch})")
ax.set_ylim(0, max(values)*1.25); ax.spines[["top","right"]].set_visible(False)

ax = axes[1]
bws = [(N*2 + M*2 + N*M*2) / ms / 1e9 for ms in values]
bars = ax.bar(labels, bws, color=colors, width=0.45, edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, bws):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(bws)*0.01,
            f"{val:.2f}", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Effective Bandwidth (TB/s)"); ax.set_title(f"Outer Add Bandwidth  (N={N}, M={M}, {arch})")
ax.set_ylim(0, max(bws)*1.25); ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.show()